# Tourist Satisfaction Analysis
Focuses on ratings-based satisfaction across destinations, cities, and location types.

In [1]:
# ============================================
# Notebook 5: Tourist Satisfaction Analysis
# ============================================
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

# --------------------------------------------
# 1. Load final dataset
# --------------------------------------------
INPUT_FILE = os.getenv("INPUT_FILE", "Processed_Reviews.csv")
MIN_REVIEWS_DEST = 20
MIN_REVIEWS_CITY = 30

df = pd.read_csv(INPUT_FILE)

print(f"Loaded: {INPUT_FILE}")
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Loaded: Processed_Reviews.csv
Shape: (16156, 41)

Columns:
['Location_Name', 'Located_City', 'Province', 'District', 'Location_Type', 'User_Locale', 'User_Country', 'User_Region', 'Travel_Date_Month', 'Travel_Date_Year', 'Published_Date_Month', 'Published_Date_Year', 'User_Contributions', 'Rating', 'Helpful_Votes', 'Review_Length', 'Title_Length', 'Rating_Class', 'Review_Delay_Days', 'Combined_Text', 'Combined_Sentiment', 'Sentiment_Score', 'Emotion', 'Dominant_Topic', 'Topic_Probability', 'Topic_Keywords', 'Review_Theme', 'Sentiment_Rating_Gap', 'Length_Bucket', 'Has_Helpful_Votes', 'Helpfulness_Ratio', 'Helpful_Bucket', 'Reviewer_Experience', 'Sentiment_Numeric', 'Location_Avg_Rating', 'Location_Review_Count', 'Location_Sentiment_Mean', 'Rank_By_Rating', 'Rank_By_Popularity', 'Popularity_Quality_Gap', 'Review_Quality_Score']


In [2]:
# --------------------------------------------
# 2. Validate required columns
# --------------------------------------------
required_cols = ["Location_Name", "Located_City", "Location_Type", "Rating"]
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

In [3]:
# --------------------------------------------
# 3. Overall satisfaction overview
# --------------------------------------------
print("\n=== Overall Rating Summary ===")
print(df["Rating"].describe())

print("\n=== Rating Distribution ===")
print(df["Rating"].value_counts().sort_index())

if "Rating_Class" in df.columns:
    print("\n=== Rating Class Distribution ===")
    print(df["Rating_Class"].value_counts())


=== Overall Rating Summary ===
count    16156.000000
mean         4.167492
std          1.006840
min          1.000000
25%          4.000000
50%          4.000000
75%          5.000000
max          5.000000
Name: Rating, dtype: float64

=== Rating Distribution ===
Rating
1     487
2     658
3    2166
4    5196
5    7649
Name: count, dtype: int64

=== Rating Class Distribution ===
Rating_Class
Positive    12845
Neutral      2166
Negative     1145
Name: count, dtype: int64


In [4]:
# --------------------------------------------
# 4. Destination-level satisfaction
# --------------------------------------------
destination_summary = (
    df.groupby("Location_Name")
      .agg(
          Review_Count=("Rating", "count"),
          Avg_Rating=("Rating", "mean"),
          Rating_STD=("Rating", "std"),
      )
      .reset_index()
 )
destination_summary["Rating_STD"] = destination_summary["Rating_STD"].fillna(0)

destination_filtered = destination_summary[
    destination_summary["Review_Count"] >= MIN_REVIEWS_DEST
].copy()

top_destinations = destination_filtered.sort_values(
    by=["Avg_Rating", "Review_Count"],
    ascending=[False, False],
).head(15)

bottom_destinations = destination_filtered.sort_values(
    by=["Avg_Rating", "Review_Count"],
    ascending=[True, False],
).head(15)

print(f"\n=== Top 15 Destinations by Average Rating (min {MIN_REVIEWS_DEST} reviews) ===")
print(top_destinations)

print(f"\n=== Bottom 15 Destinations by Average Rating (min {MIN_REVIEWS_DEST} reviews) ===")
print(bottom_destinations)


=== Top 15 Destinations by Average Rating (min 20 reviews) ===
                                Location_Name  Review_Count  Avg_Rating  \
31               Kalametiya Eco Bird Watching           132    4.992424   
8                       Bundala National Park           189    4.698413   
71                    Udawalawe National Park           149    4.677852   
11                   Community Tsunami Museum           302    4.665563   
65         Sigiriya The Ancient Rock Fortress           285    4.645614   
23                     Handunugoda Tea Estate           304    4.638158   
5                               Bentota Beach           233    4.587983   
60                    Royal Botanical Gardens           337    4.572700   
42                                  Mihintale           279    4.562724   
52                           Passikudah Beach           160    4.562500   
37                       Kumana National Park            35    4.542857   
35                  Kelaniya Raja Ma

In [5]:
# --------------------------------------------
# 5. City-level satisfaction
# --------------------------------------------
city_summary = (
    df.groupby("Located_City")
      .agg(
          Review_Count=("Rating", "count"),
          Avg_Rating=("Rating", "mean"),
          Rating_STD=("Rating", "std"),
      )
      .reset_index()
 )
city_summary["Rating_STD"] = city_summary["Rating_STD"].fillna(0)

city_filtered = city_summary[
    city_summary["Review_Count"] >= MIN_REVIEWS_CITY
].copy()

top_cities = city_filtered.sort_values(
    by=["Avg_Rating", "Review_Count"],
    ascending=[False, False],
).head(15)

bottom_cities = city_filtered.sort_values(
    by=["Avg_Rating", "Review_Count"],
    ascending=[True, False],
).head(15)

print(f"\n=== Top Cities by Average Rating (min {MIN_REVIEWS_CITY} reviews) ===")
print(top_cities)

print(f"\n=== Bottom Cities by Average Rating (min {MIN_REVIEWS_CITY} reviews) ===")
print(bottom_cities)


=== Top Cities by Average Rating (min 30 reviews) ===
     Located_City  Review_Count  Avg_Rating  Rating_STD
16     Kalametiya           132    4.992424    0.087039
35      Weligatta           189    4.698413    0.635014
10   Embilipitiya           149    4.677852    0.755775
0        Ahangama           304    4.638158    0.635052
5         Bentota           233    4.587983    0.726489
26     Peradeniya           337    4.572700    0.724710
17       Kalkudah           160    4.562500    0.688518
2          Ampara            35    4.542857    0.700540
21       Koslanda            86    4.488372    0.698640
12       Habarana           754    4.437666    0.852751
32  Tissamaharama            91    4.428571    0.652225
28    Polonnaruwa           333    4.423423    0.786362
11          Galle           511    4.422701    0.852559
14      Hikkaduwa           515    4.400000    0.928997
8        Deniyaya           196    4.331633    0.898557

=== Bottom Cities by Average Rating (min 30 revi

In [6]:
# --------------------------------------------
# 6. Satisfaction by location type
# --------------------------------------------
type_summary = (
    df.groupby("Location_Type")
      .agg(
          Review_Count=("Rating", "count"),
          Avg_Rating=("Rating", "mean"),
          Rating_STD=("Rating", "std"),
      )
      .reset_index()
      .sort_values(by="Avg_Rating", ascending=False)
 )
type_summary["Rating_STD"] = type_summary["Rating_STD"].fillna(0)
print("\n=== Satisfaction by Location Type ===")
print(type_summary)


=== Satisfaction by Location Type ===
              Location_Type  Review_Count  Avg_Rating  Rating_STD
4            Historic Sites          1519    4.388413    0.838326
6            National Parks          1205    4.385892    0.993149
8           Religious Sites          3017    4.345376    0.818413
9                Waterfalls           933    4.189711    0.839874
5                   Museums          1525    4.136393    0.935460
7   Nature & Wildlife Areas          1557    4.093128    1.183514
2                     Farms          1884    4.077495    1.026646
3                   Gardens          1354    4.070901    1.045595
0                   Beaches          2110    4.015640    1.137841
1           Bodies of Water           839    3.989273    0.940283
10       Zoological Gardens           213    3.122066    1.412255


In [7]:
# --------------------------------------------
# 7. Province-level satisfaction (if available)
# --------------------------------------------
if "Province" in df.columns:
    province_summary = (
        df.groupby("Province")
          .agg(
              Review_Count=("Rating", "count"),
              Avg_Rating=("Rating", "mean"),
              Rating_STD=("Rating", "std"),
          )
          .reset_index()
          .sort_values(by="Avg_Rating", ascending=False)
    )
    province_summary["Rating_STD"] = province_summary["Rating_STD"].fillna(0)
    print("\n=== Satisfaction by Province ===")
    print(province_summary)
else:
    province_summary = None


=== Satisfaction by Province ===
                 Province  Review_Count  Avg_Rating  Rating_STD
5       Southern Province          2648    4.408988    0.900838
2  North Central Province          3171    4.310943    0.855380
1        Eastern Province          1162    4.215146    1.028539
6            Uva Province          1040    4.170192    1.031214
3       Northern Province           475    4.141053    0.931798
0        Central Province          5252    4.073115    0.995316
7        Western Province          1890    3.932275    1.150260
4   Sabaragamuwa Province           518    3.781853    1.399329


In [8]:
# --------------------------------------------
# 8. Most stable vs most mixed destinations
# --------------------------------------------
stable_destinations = destination_filtered.sort_values(
    by=["Rating_STD", "Review_Count"],
    ascending=[True, False],
).head(15)

mixed_destinations = destination_filtered.sort_values(
    by=["Rating_STD", "Review_Count"],
    ascending=[False, False],
).head(15)

print(f"\n=== Most Stable Destinations (low rating variation, min {MIN_REVIEWS_DEST} reviews) ===")
print(stable_destinations)

print(f"\n=== Most Mixed Destinations (high rating variation, min {MIN_REVIEWS_DEST} reviews) ===")
print(mixed_destinations)


=== Most Stable Destinations (low rating variation, min 20 reviews) ===
                                Location_Name  Review_Count  Avg_Rating  \
31               Kalametiya Eco Bird Watching           132    4.992424   
42                                  Mihintale           279    4.562724   
11                   Community Tsunami Museum           302    4.665563   
41  Martin Wickramasinghe Folk Museum Complex            86    4.511628   
8                       Bundala National Park           189    4.698413   
23                     Handunugoda Tea Estate           304    4.638158   
69                                 Tissa Wewa            91    4.428571   
59                  Ritigala Forest Monastery           132    4.446970   
52                           Passikudah Beach           160    4.562500   
17                                 Galle Fort           283    4.462898   
16                             Diyaluma Falls            86    4.488372   
47                         